[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/piezoelectric/piezoelectric.ipynb)

# Piezoelectric Bimorph

This notebook computes the mechanical deflection, internal strain and terminal response of a laminated piezoelectric bimorph under applied voltage and tip force. Two oppositely poled ceramic layers share a central bond between top and bottom metal electrodes, with one end held in a rigid clamp. Driving the electrodes makes one layer lengthen and the other contract so the strip bends as an actuator, while pushing the tip polarizes both layers so the same strip acts as a charge or voltage sensor.

Planar spacetime gives each physical quantity its natural grade and orientation without auxiliary vector spaces. Displacements sit on intrinsic spatial directions, electric fields are extrinsic spacetime bivectors formed from potential slopes as `gradient ^ potential`, and dielectric and piezoelectric excitations live on intrinsic antibivectors swept through time (`direction & time`) that pair directly with the field via `field & excitation`. Pulling the triangle work pairings back through the corner incidences builds one off-diagonal coupling map whose `.adjugate()` supplies the converse sensor charge from the actuator force law.

In Voigt matrix-tensor notation for linear piezoelectricity, in which [Martins and Cordioli](https://dael.euracoustics.org/bin/EAA/aaua_dl?document_id=52105) give the plane-stress PZT-5A data used here and the [ANSYS piezoelectric bimorph example](https://ansyshelp.ansys.com/public/views/secured/corp/v242/en/ans_cou/Hlp_G_COU3_piezo.html#ajjs121101406) sets up the same actuator and sensor, symmetric strain components are packed into a 3-component engineering vector $\boldsymbol{\varepsilon} = [\partial_x u_x, \partial_y u_y, \partial_x u_y + \partial_y u_x]^\mathsf{T}$ with factor-of-two shear conventions, coupled to electric field components $\mathbf{E} = -\nabla\phi$ through reduced $3\times 3$, $2\times 3$ and $2\times 2$ coordinate tables $\boldsymbol{\sigma} = \mathbf{c}^E\boldsymbol{\varepsilon} - \mathbf{e}^\mathsf{T}\mathbf{E}$ and $\mathbf{D} = \mathbf{e}\boldsymbol{\varepsilon} + \boldsymbol{\epsilon}^S\mathbf{E}$ that transform by separate Bond rotation matrices when the poling axis turns.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"
%matplotlib inline

In [ ]:
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import STA2D as ga
from numga.sparse import Sparse, SparseExtensor, spdiag
from numga.topology import Topology

from examples.electromagnetism.piezoelectric import render, scenarios
from examples.electromagnetism.piezoelectric.numerical import solve_coupled
from examples.util.animation import save_figure, show_states

# Planar spacetime x+y+t-: two positive spatial axes and one negative time axis.
ctx = NumpyContext(ga)
mv = ctx.multivector
Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector().derive.within(Algebra("x+y+"))
Point = Plane.derive.dual
Field = ga.gatype.bivector().derive.touching(Algebra("t-"))
Excitation = Field.derive.dual
Strain = ga.gatype((Plane, Point))
Elastic = ga.gatype((Scalar, Point, Plane, Point, Plane))
Polarization = ga.gatype((Excitation, Point, Plane))
Permittivity = ga.gatype((Excitation, Field))

time = -mv.t.dual()                                                               # Antivector-
axis = mv.x.dual()                                                                # Point-
pole = mv.y.dual()                                                                # Point-

## Laminated strip

Two PZT-5A ceramic layers meet along a central bond with their poling directions pointing apart, clamped at the left end and plated by top and bottom electrodes. Small deflections and zero through-width stress govern the slender strip.

In [ ]:
topology, coordinates, layer_poling, top_weights, tip_weights = scenarios.bimorph(
    scenarios.length, scenarios.thickness, scenarios.n_length, scenarios.n_thickness,
)
vertices = mv(Plane, coordinates).dual().field()                                  # Point-[V]
poling = mv.scalar(layer_poling).field()                                          # Scalar+[F]
top_electrode = mv.scalar(top_weights).field()                                    # Scalar+[V]
tip_distribution = mv.scalar(tip_weights).field()                                 # Scalar+[V]
save_figure(render.draw_strip(topology, vertices), "piezoelectric_strip");

Near the hatched left clamp, dark markers show the pinned boundary vertices, orange and blue strokes mark the top and bottom electrodes, and the slate line along the midplane marks the bonded interface between the two ceramic layers.

## Simplex gradients and triangle strain

Each triangle's boundary cycle orients the edge vector facing each corner, and joining two sides against `time` yields twice the signed area as an intrinsic scalar that reverses under mirror reflection. The term ***simplex gradient*** here means each corner's spatial slope across the triangle, formed as `(time & sides) / double_areas`, from which the symmetrized sum of corner displacements builds the local strain map.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Scalar, Plane]:
    """Opposite edge vectors joined against time give each triangle's area (`Scalar+[F]`) and corner simplex gradients (`[3] Plane+[F]`)."""
    # Each corner's opposite edge vector, oriented along the triangle's boundary cycle.
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                  # [3] Point-[F]
    double_areas = (sides[0] & sides[1]) & time                                   # Antiscalar-[F]
    # Sweeping each opposite side through time and dividing by twice the signed area gives the corner's simplex gradient.
    gradients = (time & sides) / double_areas                                     # [3] Plane+[F]
    return double_areas.norm() / 2, gradients


def strain(topology: Topology, displacement: Point, gradients: Plane) -> Strain:
    """Corner displacements paired with the simplex gradients form the symmetric strain map (`[F] Plane+ <- Point-`) on each triangle."""
    # Evaluate on the batch of triangles so the adjugate acts on each triangle's local map.
    corner_displacements = (topology.incidence[2, 0] * displacement).batch()      # [..., 3, F] Point-
    gradient = (corner_displacements.dual_inverse() * (gradients.batch() & Point)).sum(axis=-2)  # [..., F] Plane+ <- Point-
    return (gradient + gradient.adjugate()) / 2


areas, gradients = triangles(topology, vertices)
zero_motion = 0 * vertices                                                        # Point-[V]
zero_potential = 0 * top_electrode                                                # Scalar+[V]
zero_strain = strain(topology, zero_motion, gradients)(axis) & axis               # [F] Scalar+
save_figure(
    render.draw_device(
        topology, vertices, areas, zero_motion, zero_potential, zero_strain,
        scenarios.displacement_scale, scenarios.applied_voltage, scenarios.strain_scale,
    ),
    "piezoelectric_setup",
);

In the unloaded reference configuration, chevron arrows across the upper and lower halves point outward from the central bond to show the opposite poling of the two layers.

## Constitutive coupling and electric enthalpy

Two orthogonal material planes specify the crystal axes transverse and parallel to poling, while their spatial directions swept through `time` form the corresponding excitation world-sheets. Extension and shear readings build the elastic work form and the piezoelectric excitation map so turning or reflecting those planes transforms the full electromechanical response automatically, and subtracting the electrical pairings `electric_field & piezo_excitation` and `(electric_field & dielectric_excitation) / 2` accounts for work exchanged with a controlled-voltage source.

In [ ]:
def constitutive(directions: Plane, stretch_stiffnesses: np.ndarray, cross_stiffness: float,
                 shear_stiffness: float, stretch_polarizations: np.ndarray,
                 shear_polarization: float, permittivities: np.ndarray
                 ) -> tuple[Elastic, Polarization, Permittivity]:
    """Two orthogonal material planes (`[...] Plane+`) determine the elastic, piezoelectric and permittivity maps."""
    transverse_plane, poling_plane = directions[..., 0], directions[..., 1]       # [...] Plane+, [...] Plane+
    sheets = directions.dual() & time                                             # [..., 2] Excitation-
    transverse_sheet, poling_sheet = sheets[..., 0], sheets[..., 1]               # [...] Excitation-, [...] Excitation-
    # Each material plane reads extension along its own normal direction; their symmetric cross reading is shear.
    stretches = (directions & Point) * (Plane & directions.dual())                # [..., 2] Scalar+ <- (Point-, Plane+)
    shear = (
        (transverse_plane & Point) * (Plane & poling_plane.dual())
        + (poling_plane & Point) * (Plane & transverse_plane.dual())
    )                                                                             # [...] Scalar+ <- (Point-, Plane+)
    elastic = (
        (stretches * stretches * stretch_stiffnesses).sum(axis=-1)
        + cross_stiffness * (stretches[..., 0] * stretches[..., 1] + stretches[..., 1] * stretches[..., 0])
        + shear_stiffness * shear * shear
    )
    # Extension excites polarization along the poling world-sheet; shear excites it across the transverse world-sheet.
    polarization = (
        poling_sheet * (stretches * stretch_polarizations).sum(axis=-1)
        + transverse_sheet * shear_polarization * shear
    )
    permittivity = (sheets * (Field & sheets) * permittivities).sum(axis=-1)
    return elastic, polarization, permittivity


def enthalpy(local_strain: Strain, electric_field: Field, elastic: Elastic, polarization: Polarization,
             permittivity: Permittivity, poling: Scalar) -> Scalar:
    """Local strain (`[...] Plane+ <- Point-`) and electric field (`[...] Field+`) contract with the material maps into the electric enthalpy density (`[...] Scalar+`)."""
    deformation = local_strain.dual()                                             # [...] Point- <- Point-
    # Each trace contracts one displacement-gradient pair of the constitutive map with the local deformation.
    elastic_density = elastic(deformation, Plane, deformation, Plane).trace(1, 2).trace(1, 2) / 2  # [...] Scalar+
    piezo_excitation = polarization(deformation, Plane).trace(1, 2) * poling      # [...] Excitation-
    dielectric_excitation = permittivity(electric_field)                          # [...] Excitation-
    return elastic_density - (electric_field & piezo_excitation) - (electric_field & dielectric_excitation) / 2


directions = stack((mv.x, mv.y))                                                  # [2] Plane+
elastic, polarization, permittivity = constitutive(
    directions, scenarios.stretch_stiffnesses, scenarios.cross_stiffness, scenarios.shear_stiffness,
    scenarios.stretch_polarizations, scenarios.shear_polarization, scenarios.permittivities,
)

## Coupled equilibrium assembly

On each triangle, the corner simplex gradients turn vertex displacements into local strain and vertex potentials into an electric field `corner_gradients ^ (-t * Scalar)`. Pulling the three enthalpy blocks back through `corners.adjugate()` sums triangle forces and induced charges onto the vertices, with a single off-diagonal `coupling` map whose `.adjugate()` supplies charge from motion.

In [ ]:
def assemble(topology: Topology, areas: Scalar, gradients: Plane, elastic: Elastic, polarization: Polarization,
             permittivity: Permittivity, poling: Scalar, width: float
             ) -> tuple[SparseExtensor, SparseExtensor, SparseExtensor]:
    """Pullback through the corner incidences sums the triangle enthalpy blocks into vertex maps (`[V, V]`)."""
    # Bind each corner's simplex gradient to form triangle-level work and excitation maps (test corner first, trial corner second).
    corner_gradients = gradients.batch()                                          # [3, F] Plane+
    work = elastic(Point, corner_gradients[None], Point, corner_gradients[:, None])  # [3, 3, F] Scalar+ <- (Point-, Point-)
    piezo_map = polarization(Point, corner_gradients) * poling.batch()            # [3, F] Excitation- <- Point-
    field_map = corner_gradients ^ (-mv.t * Scalar)                               # [3, F] Field+ <- Scalar+

    # Convert the bilinear work form into a linear map from trial displacement to test force.
    elastic_blocks = (Plane & Point).solve(work)                                  # [3, 3, F] Plane+ <- Point-
    # Pulling excitation back across the electric field pairs trial voltage with test force;
    # its adjugate supplies induced charge from displacement without a second coupling law.
    mixed_blocks = -piezo_map[:, None].adjugate()(field_map[None])                # [3, 3, F] Plane+ <- Scalar+
    electric_blocks = -field_map[:, None].adjugate()(permittivity(field_map[None]))  # [3, 3, F] Antiscalar- <- Scalar+

    volumes = width * areas.batch()                                               # [F] Scalar+
    corners = topology.incidence[2, 0]                                            # [3] Scalar+[F <- V]
    return tuple(
        (corners[:, None].adjugate() * spdiag((local_blocks * volumes).field()) * corners[None, :]).sum(axis=0).sum(axis=0)
        for local_blocks in (elastic_blocks, mixed_blocks, electric_blocks)
    )


def equilibrium(mechanical: SparseExtensor, coupling: SparseExtensor, dielectric: SparseExtensor,
                free_motion: Sparse, free_voltage: Sparse,
                electrode_potential: Scalar, mechanical_load: Plane) -> tuple[Point, Scalar]:
    """Mechanical equilibrium and charge balance determine the vertex displacements (`Point-[V]`) and potentials (`Scalar+[V]`)."""
    # Restrict the enthalpy blocks to unconstrained displacements and independent potential groups.
    elastic = free_motion.adjugate() * mechanical * free_motion
    mixed = free_motion.adjugate() * coupling * free_voltage
    capacitance = free_voltage.adjugate() * dielectric * free_voltage
    load = free_motion.adjugate() * (mechanical_load - coupling(electrode_potential))
    charge = -(free_voltage.adjugate() * dielectric(electrode_potential))
    displacement, potential = solve_coupled(elastic, mixed, capacitance, load, charge)
    return free_motion * displacement, electrode_potential + free_voltage * potential


mechanical, coupling, dielectric = assemble(
    topology, areas, gradients, elastic, polarization, permittivity, poling, scenarios.width,
)

## Voltage actuation

Applying 100 V to the top electrode against a grounded bottom electrode with zero mechanical tip load puts a downward electric field across both layers. Because the upper layer is poled upward and the lower layer downward with negative transverse coupling, the upper layer lengthens axially while the lower layer contracts, curling the free cantilever downward.

In [ ]:
free_motion, driven_voltage = scenarios.boundary_maps(topology, np.array([], dtype=int))
actuator_motion, actuator_potential = equilibrium(
    mechanical, coupling, dielectric, free_motion, driven_voltage,
    scenarios.applied_voltage * top_electrode, 0 * mv.y * tip_distribution,
)
actuator_strain = strain(topology, actuator_motion, gradients)(axis) & axis        # [F] Scalar+
save_figure(
    render.draw_device(
        topology, vertices, areas, actuator_motion, actuator_potential, actuator_strain,
        scenarios.displacement_scale, scenarios.applied_voltage, scenarios.strain_scale,
    ),
    "piezoelectric_actuator",
);

In [ ]:
def load_cycle(displacement: Point, potential: Scalar, axial_strain: Scalar,
               n_frames: int) -> Iterator[tuple[Point, Scalar, Scalar]]:
    """Yields sinusoidal equilibrium states (`Point-[V], Scalar+[V], Scalar+[F]`) across one drive period."""
    for phase in np.linspace(0, 2 * np.pi, n_frames, endpoint=False):
        amplitude = float(np.sin(phase))
        yield displacement * amplitude, potential * amplitude, axial_strain * amplitude


actuator_states = show_states(
    load_cycle(actuator_motion, actuator_potential, actuator_strain, scenarios.n_frames),
    render.animate, topology, vertices, areas,
    scenarios.displacement_scale, scenarios.applied_voltage, scenarios.strain_scale,
    name="piezoelectric_actuator", duration_ms=scenarios.frame_duration_ms, n_frames=scenarios.n_frames,
)

Orange shading inside the strip indicates axial extension and blue shading indicates axial compression, while the coloured top and bottom borders show the signed terminal voltage over a 100-fold magnification of the displacement relative to the dashed rest outline.

In [ ]:
actuator_tip = ((mv.y & actuator_motion) * tip_distribution).sites.sum()          # Scalar+
tip_micrometres = actuator_tip * scenarios.length_unit * 1e6                      # Scalar+
print(f"Tip displacement at 100 V: {tip_micrometres.to_array():.2f} μm")

## Open-circuit sensing

A 60 mN upward transverse force distributed across the tip bends the cantilever while the bottom electrode stays grounded and the top electrode floats at zero net free charge. Grouping all top-electrode vertices onto one unknown potential solves simultaneously for the induced open-circuit voltage and its stiffening feedback on the mechanical deflection.

In [ ]:
_, floating_voltage = scenarios.boundary_maps(topology, topology.top_nodes)
sensor_motion, sensor_potential = equilibrium(
    mechanical, coupling, dielectric, free_motion, floating_voltage,
    0 * top_electrode, (mv.y * scenarios.applied_force) * tip_distribution,
)
sensor_strain = strain(topology, sensor_motion, gradients)(axis) & axis           # [F] Scalar+
save_figure(
    render.draw_device(
        topology, vertices, areas, sensor_motion, sensor_potential, sensor_strain,
        scenarios.displacement_scale, scenarios.sensor_voltage_scale, scenarios.strain_scale,
    ),
    "piezoelectric_sensor",
);

In [ ]:
sensor_states = show_states(
    load_cycle(sensor_motion, sensor_potential, sensor_strain, scenarios.n_frames),
    render.animate, topology, vertices, areas,
    scenarios.displacement_scale, scenarios.sensor_voltage_scale, scenarios.strain_scale,
    name="piezoelectric_sensor", duration_ms=scenarios.frame_duration_ms, n_frames=scenarios.n_frames,
)

Under tip bending, axial strain peaks near the clamped root and vanishes toward the free tip, while the electrode colour scale spans ±10 V between the floating top plate and the grounded bottom plate.

In [ ]:
top = topology.reading(topology.top_nodes, 0)
sensor_volts = (top * sensor_potential).sites.mean() * scenarios.voltage_unit     # Scalar+
print(f"Open-circuit voltage at 60 mN: {sensor_volts.to_array():.2f} V")

## Reciprocal terminal response

Shorting both electrodes to ground while applying the same tip load lets bending drive charge through the external wire. Because actuation forces and induced charges come from `coupling` and `coupling.adjugate()`, the short-circuit charge per unit tip force equals the actuator tip displacement per unit volt, and dividing the short-circuit charge by the free-strip capacitance recovers the open-circuit sensor voltage.

In [ ]:
short_motion, short_potential = equilibrium(
    mechanical, coupling, dielectric, free_motion, driven_voltage,
    0 * top_electrode, (mv.y * scenarios.applied_force) * tip_distribution,
)
# Free charge on the top electrode is the negative potential derivative of electric enthalpy.
actuator_charges = -(coupling.adjugate()(actuator_motion) + dielectric(actuator_potential))  # Antiscalar-[V]
short_charges = -(coupling.adjugate()(short_motion) + dielectric(short_potential))  # Antiscalar-[V]
actuator_charge = (top_electrode & actuator_charges).sites.sum()                  # Scalar+
short_charge = (top_electrode & short_charges).sites.sum()                        # Scalar+
capacitance = actuator_charge / scenarios.applied_voltage                         # Scalar+
actuator_gain = actuator_tip / scenarios.applied_voltage                          # Scalar+
sensor_gain = short_charge / scenarios.applied_force                              # Scalar+
predicted_voltage = -short_charge / capacitance                                   # Scalar+

print(f"Actuator sensitivity:   {float((actuator_gain * scenarios.length_unit / scenarios.voltage_unit * 1e9).to_array()):.2f} nm/V")
print(f"Charge sensitivity:     {float((sensor_gain * scenarios.charge_unit / scenarios.force_unit * 1e9).to_array()):.2f} nC/N")
print(f"Free capacitance:       {float((capacitance * scenarios.charge_unit / scenarios.voltage_unit * 1e9).to_array()):.2f} nF")
print(f"Predicted open voltage: {float((predicted_voltage * scenarios.voltage_unit).to_array()):.2f} V")

In [ ]:
# checks
assert len(actuator_states) == scenarios.n_frames and len(sensor_states) == scenarios.n_frames
assert (zero_strain.abs() < 1e-12).all()
assert ((topology.reading(topology.clamp_nodes, 0) * actuator_motion).antinorm() < 1e-12).all()
assert ((topology.reading(topology.clamp_nodes, 0) * sensor_motion).antinorm() < 1e-12).all()
assert (actuator_gain - sensor_gain).abs() < 1e-9 * actuator_gain.abs()
sensor_charges = -(coupling.adjugate()(sensor_motion) + dielectric(sensor_potential))
assert (top_electrode & sensor_charges).sites.sum().abs() < 1e-10 * short_charge.abs()
assert (sensor_volts / scenarios.voltage_unit - predicted_voltage).abs() < 1e-9 * predicted_voltage.abs()
actuator_field = ((topology.incidence[2, 0] * actuator_potential).batch() * (gradients.batch() ^ (-mv.t))).sum(axis=-2)
total_enthalpy = (
    scenarios.width * areas.batch()
    * enthalpy(strain(topology, actuator_motion, gradients), actuator_field, elastic, polarization, permittivity, poling.batch())
).sum()
assert (total_enthalpy + actuator_charge * scenarios.applied_voltage / 2).abs() < 1e-9 * total_enthalpy.abs()